In [1]:
from pathlib import Path
import cv2
import numpy as np
from mmdet.apis import inference_detector, init_detector

In [2]:
config_path = Path(r"C:\dev\projects\CV_counting_bags\configs\rtmdet_tiny_bag.py")
ckpt_path = Path(r"C:\dev\projects\CV_counting_bags\work_dirs\rtmdet_tiny_bag\best_coco_bbox_mAP_epoch_27.pth")
video_path = Path(r"C:\dev\projects\CV_counting_bags\input.mp4")
out_video_path = Path(r"C:\dev\projects\CV_counting_bags\output_video\output.mp4")

In [3]:
model = init_detector(
    str(config_path),
    str(ckpt_path),
    device = "cuda:0"
)

Loads checkpoint by local backend from path: C:\dev\projects\CV_counting_bags\work_dirs\rtmdet_tiny_bag\best_coco_bbox_mAP_epoch_27.pth


In [4]:
conveyor_roi = np.array([
    [0, 360],
    [310, 28],
    [484, 65],
    [315, 360],    
], dtype=np.int32)

In [5]:
def interpolate(p1, p2, coeff):
    x = p1[0] + (p2[0] - p1[0]) * coeff
    y = p1[1] + (p2[1] - p1[1]) * coeff

    return int(x), int(y)

In [6]:
line_pos_coeff = 0.35

left_point = interpolate(conveyor_roi[1], conveyor_roi[0], line_pos_coeff)
right_point = interpolate(conveyor_roi[2], conveyor_roi[3], line_pos_coeff)

counting_line = (left_point, right_point)

In [7]:
def bbox_iou(bbox1, bbox2):
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])

    inter_width = max(0, x2 - x1)
    inter_height = max(0, y2 - y1)

    area1 = max(0, bbox1[2] - bbox1[0]) * max(0, bbox1[3] - bbox1[1])
    area2 = max(0, bbox2[2] - bbox2[0]) * max(0, bbox2[3] - bbox2[1])

    intersection = inter_width * inter_height
    union = area1 + area2 - intersection
    
    if union == 0:
        return 0.0

    return intersection / union

In [ ]:
class Track:
    def __init__(self, track_id, bbox, score):
        self.id = track_id
        self.bbox = np.array(bbox, dtype = float)
        self.score = float(score)

        self.hits = 1
        self.missed = 0
        self.age = 1
        self.confirmed = False
        self.history = [self.center()]

        self.last_side = None
        self.moving = False
    
    def center(self):
        x1, y1, x2, y2 = self.bbox

        return(float((x1 + x2) / 2), float((y1 + y2) / 2))

    def update(self, bbox, score):
        self.bbox = np.array(bbox, dtype = float)
        self.score = float(score)

        self.hits += 1
        self.missed = 0
        self.age += 1
        self.history.append(self.center())

    def mark_missed(self):
        self.missed += 1
        self.age += 1

In [9]:
class IoUTracker:
    def __init__(self, iou_threshold = 0.3, max_missed = 3, min_hits = 3):
        self.iou_threshold = iou_threshold
        self.max_missed = max_missed
        self.min_hits = min_hits

        self.tracks = []
        self.next_id = 1

    def create_track(self, detection):
        track = Track(track_id = self.next_id, bbox = detection['bbox'], score = detection['score'])
        
        self.next_id += 1
        self.tracks.append(track)

    def update(self, detections):
        if len(self.tracks) == 0:
            for detection in detections:
                self.create_track(detection)

            self.update_accepted()
            return self.tracks

        if len(detections) == 0:
            for track in self.tracks:
                track.mark_missed()

                self.remove_dead_tracks()
                self.update_accepted()
                return self.tracks

        iou_matrix = np.zeros((len(self.tracks), len(detections)), dtype = float)

        for track_index, track in enumerate(self.tracks):
            for detection_index, detection in enumerate(detections):
                iou_matrix[track_index, detection_index] = bbox_iou(track.bbox, detection['bbox'])

        matched_tracks = set()
        matched_detections = set()

        while True:
            track_index, detection_index = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)

            best_iou = iou_matrix[track_index, detection_index]
            if best_iou < self.iou_threshold:
                break

            track = self.tracks[track_index]
            detection = detections[detection_index]

            track.update(detection['bbox'], detection['score'])

            matched_tracks.add(track_index)
            matched_detections.add(detection_index)

            iou_matrix[track_index, :] = -1
            iou_matrix[:, detection_index] = -1

        for detection_index, track in enumerate(self.tracks):
            if track_index not in matched_tracks:
                track.mark_missed()

        for detection_index, detection in enumerate(detections):
            if detection_index not in matched_detections:
                self.create_track(detection)

        self.remove_dead_tracks()
        self.update_accepted()

        return self.tracks

    def remove_dead_tracks(self):
        self.tracks = [track for track in self.tracks if track.missed <= self.max_missed]


    def update_accepted(self):
        for track in self.tracks:
            if track.hits >= self.min_hits:
                track.confirmed = True            

In [27]:
class AnomalyMonitor:
    def __init__(self):
        self.anomalies = []

    def add(self, name, frame_index, fps, track_id):
        anomaly = {
            "type": name,
            "frame": frame_index,
            "time_ code": frame_index / fps,
            "id": track_id,
        }
        
        self.anomalies.append(anomaly)

        return anomaly

In [ ]:
class LineCounter:
    def __init__(self, line):
        self.line = line
        self.count = 0

    def point_side(self, point):
        x, y = point
        (x1, y1), (x2, y2) = self.line

        value = ((x2 - x1) * (y - y1) - (y2 - y1) * (x - x1))

        if value > 0:
            return 1
        elif value < 0:
            return -1
        else:
            return 0

    def update(self, track):
        curr_side = self.point_side(track.center())

        if curr_side == 0:
            return None
            
        if track.last_side is None:
            track.last_side = curr_side
            return None

        anomaly = None

        if track.last_side == 1 and curr_side == -1:
            self.count += 1

        if track.last_side == -1 and curr_side == 1:
            self.count -= 1
            anomaly = "reverse movement"

        track.last_side = curr_side

        return anomaly

In [11]:
def inside_roi(obj, roi):
    obj_center = (int((obj[0] + obj[2]) / 2), int((obj[1] + obj[3]) / 2))
    result = cv2.pointPolygonTest(roi, obj_center, measureDist = False)

    if result >= 0:
        return True

In [12]:
def draw_detection(frame, result, roi, threshold = 0.36):
    image = frame.copy()

    pred = result.pred_instances
    bboxes = pred.bboxes.detach().cpu().numpy()
    scores = pred.scores.detach().cpu().numpy()

    detections_sum = 0
    track_detections = []

    for bbox, score in zip(bboxes, scores):
        if score < threshold:
            continue

        if inside_roi(bbox, roi):
            track_detections.append({
                "bbox": bbox,
                "score": score
            })

        detections_sum += 1

        x1, y1, x2, y2 = bbox.astype(int)

        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"bag {score:.2f}"

        cv2.putText(image, label, (x1, max(y1 - 7, 20)), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.putText(image, f"Detections: {detections_sum}",
        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    cv2.putText(image, f"Tracking candidates: {len(track_detections)}",
        (20, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    #cv2.line(image, counting_line[0],
     #   counting_line[1], (0, 255, 255), 2)

    return image, track_detections

In [ ]:
def process_video(viedo_path, out_path, model, threshold = 0.36, max_frames = None):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError("failed to open video:", video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"FPS: {fps}")
    print(f"Res: {width}x{height}")
    print(f"Frames: {frames_count}")

    out_path = Path(out_path)
    out_path.parent.mkdir(parents = True, exist_ok = True)
    
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))

    if not writer.isOpened():
        cap.release()
        raise RuntimeError("failed to create video", out_path)
        
    tracker = IoUTracker(iou_threshold = 0.3, max_missed = 3, min_hits = 3)
    
    frame_index = 0
    counter = LineCounter(counting_line)
    monitor = AnomalyMonitor()
    
    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if max_frames is not None and frame_index >= max_frames:
            break

        result = inference_detector(model, frame)
        annotation, track_detections = draw_detection(frame, result, conveyor_roi)
        tracks = tracker.update(track_detections)

        for track in tracks:
            if not track.confirmed:
                continue
            
            x1, y1, x2, y2 = track.bbox.astype(int)

            anomaly = counter.update(track)
            if anomaly is not None:
                monitor.add(anomaly, frame_index, fps, track.id)
                
                #anomaly_time = 10 * fps
                
                #while anomaly_time > 0:
                 #   cv2.putText(annotation, f"anomaly detected: {anomaly}", (20, 150),
                  #      cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

                   # anomaly_time -= 1
            
            cv2.putText(annotation, f"ID {track.id}", (x1, y2 + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        cv2.putText(annotation, f"Bags count: {counter.count}",
            (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
            
        writer.write(annotation)
        frame_index += 1

    cap.release()
    writer.release()
    print("done")
    
    return {
        "count": counter.count,
        "anomalies": monitor.anomalies
    }

In [ ]:
process_video(video_path, out_path = out_video_path, model = model, max_frames = 7500)